In [ ]:
import numpy as np 
import os 
import cv2 

original_frame = cv2.imread('./t.png', cv2.IMREAD_GRAYSCALE)
img_cpy = cv2.imread('./t.png', cv2.IMREAD_GRAYSCALE)

frame = cv2.GaussianBlur(original_frame,(25,25),0)
ret,thresh = cv2.threshold(frame,85,200,cv2.THRESH_BINARY)
cv2.imwrite("threshold.png", thresh)

element = cv2.getStructuringElement(cv2.MORPH_CROSS, (7,7))
element2 = cv2.getStructuringElement(cv2.MORPH_CROSS, (15,15))
dil_el = cv2.getStructuringElement(cv2.MORPH_CROSS, (3,3))
eroded = cv2.erode(thresh, element)
for _ in range(20):
    eroded = cv2.erode(eroded, dil_el)
    eroded = cv2.dilate(eroded, element)
cv2.imwrite("eroded.png", eroded)

eroded = cv2.medianBlur(eroded,175,0)

edges = cv2.Canny(eroded, 200, 300, apertureSize=3)
# edges = cv2.dilate(edges, element2)
# edges = cv2.erode(edges, element2)
cv2.imwrite('edge3.png', edges)

lines = cv2.HoughLinesP(edges, 1, np.pi/90, 50, minLineLength=100, maxLineGap=5)

print(lines)
line_data = []
for line in lines:
    x1, y1, x2, y2 = line[0]

    cv2.line(img_cpy, (x1, y1), (x2, y2), (0, 0, 255), 2)


    change_in_x = x2 - x1
    change_in_y = y2 - y1
    angle = np.arctan2(change_in_y, change_in_x)             
    theta = (angle + np.pi / 2) % np.pi    
    r = x1 * np.cos(angle) + y1 * np.sin(angle)

    exists = False
    for i, (other_r, other_theta) in enumerate(line_data):
        if abs(r-other_r) < 300 and abs(theta-other_theta) < 5: # I just chose these based on the image I was exploring... Probably a better way for finding these
            line_data[i] = ((r+other_r)/2, (theta+other_theta)/2)
            exists = True
            break

    if not exists:
        line_data.append((r, theta))
    print(f"r = {r}, theta = {np.degrees(angle)} degrees")

    angle_deg = np.degrees(angle)
    cv2.putText(img_cpy, f"{angle_deg} degrees", (x1, y1), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)

cv2.imwrite('linesDetected.png', img_cpy)

[[[ 232  993  381  951]]

 [[1271  873 1410  935]]

 [[  37 1050  138 1021]]

 [[1359  912 1464  959]]

 [[1129  798 1230  851]]

 [[ 389  950  493  921]]]
r = 15.21, θ = -15.74°
r = 11.69, θ = 24.04°
r = 8.95, θ = -16.02°
r = 6.84, θ = 24.11°
r = 6.83, θ = 27.69°
r = 4.97, θ = -15.58°
[(6.446094169493895, 1.5952991969270112)]
Slope: 0.02, Angle: 1.38°


True